# 🤖 Lesson 12: Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/12_generation.ipynb)

> **Retrieval finds the right pages. Generation turns them into an answer, *if* we ask the model the right way.**

---

## 🎯 Goal
By the end of this lesson you can build a prompt that makes the model answer only from the retrieved chunks, cite them, and say "I don't know" when the answer isn't there.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~25 minutes |
| 📚 **Prerequisites** | [Lesson 11](11_retrieval_basics.ipynb) |
| 🔑 **API key** | Required (OpenRouter, for the chat model) |
| 🛠 **Libraries** | `langchain`, `langchain-openrouter`, `chromadb`, `sentence-transformers`, `numpy` |

In [1]:
!pip install -q chromadb sentence-transformers numpy "langchain>=1" langchain-openrouter

^C


In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import chromadb

EMBED_MODEL = "intfloat/multilingual-e5-small"
embedder = SentenceTransformer(EMBED_MODEL)

chunks = [
    {"id": "pto-0", "section": "PTO", "text": "Full-time employees receive paid time off based on years of service."},
    {"id": "pto-1", "section": "PTO", "text": "With less than 1 year of service, employees receive 10 PTO days per year."},
    {"id": "pto-2", "section": "PTO", "text": "With 1 to 3 years of service, employees receive 15 PTO days per year."},
    {"id": "pto-3", "section": "PTO", "text": "With more than 3 years of service, employees receive 20 PTO days per year."},
    {"id": "pto-4", "section": "PTO", "text": "PTO requests are submitted in advance through the HR system, unless related to an emergency."},
    {"id": "conduct-0", "section": "Conduct", "text": "Employees must maintain a respectful, professional environment free from harassment, discrimination and intimidation."},
    {"id": "conduct-1", "section": "Conduct", "text": "Disruptive behavior and verbal abuse are prohibited and may lead to disciplinary action."},
    {"id": "travel-0", "section": "Travel", "text": "Reasonable and necessary expenses of approved business trips are reimbursed."},
    {"id": "travel-1", "section": "Travel", "text": "Flights, hotels and meals must be recorded with receipts in the finance system."},
    {"id": "nda-0", "section": "NDA", "text": "Employees must protect confidential information such as product roadmaps, customer data and proprietary algorithms."},
    {"id": "nda-1", "section": "NDA", "text": "Confidential information must not be shared with unauthorized people, during or after employment."},
]

chunk_vectors = embedder.encode(["passage: " + chunk["text"] for chunk in chunks], normalize_embeddings=True)

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_or_create_collection(
    name="handbook_small",
    configuration={"hnsw": {"space": "cosine"}},
)
collection.upsert(
    ids=[chunk["id"] for chunk in chunks],
    embeddings=chunk_vectors.tolist(),
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[{"source": "pdf1.pdf", "section": chunk["section"], "position": position} for position, chunk in enumerate(chunks)],
)

def retrieve(question, top_k=3, where=None):
    question_vector = embedder.encode(["query: " + question], normalize_embeddings=True)[0]
    result = collection.query(
        query_embeddings=[question_vector.tolist()],
        n_results=top_k,
        where=where,
    )
    return [
        {"id": chunk_id, "section": metadata["section"], "similarity": 1 - distance, "text": text}
        for chunk_id, metadata, distance, text in zip(
            result["ids"][0], result["metadatas"][0], result["distances"][0], result["documents"][0]
        )
    ]

def format_context(items):
    return "\n".join(f'[{item["id"]}] {item["text"]}' for item in items)

print("stored:", collection.count())

d:\AI Source\Rag_torurial\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1229.64it/s]


stored: 11


In [3]:
from getpass import getpass
import os
from langchain.chat_models import init_chat_model

if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openrouter",
    temperature=0,
)

## ✍️ Designing the prompt

We send two messages:

| Message | Contains | Purpose |
|---|---|---|
| **system** | the **rules** | tells the model *how* to behave, every time |
| **user** | the **context** (retrieved chunks) and the **question** | the material for this answer |

Our rules:
1. Use **only** the context.
2. **Cite** the ids of the chunks used, like `[pto-3]`.
3. If the context doesn't contain the answer, say **exactly** `I don't know based on the handbook.`

Why a fixed sentence for "I don't know"? Because our code can then **detect** it (`"I don't know" in answer`) instead of guessing.

We also set `temperature=0`, so the model is as repeatable as possible. (It still doesn't guarantee identical answers every time.)

In [4]:
SYSTEM_PROMPT = (
    "You answer questions about a company handbook. "
    "Use ONLY the context provided. "
    "Cite the ids of the chunks you used in square brackets, like [pto-3]. "
    "If the context does not contain the answer, reply exactly: I don't know based on the handbook."
)

def answer(question, top_k=4):
    items = retrieve(question, top_k=top_k)
    respone = model.invoke(
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{format_context(items)}\n\nQuestion: {question}"},
        ]
    )
    return respone.content, items

question = "How many PTO days do I get after five years, and how do I request them?"
text, items = answer(question)

print("retrieved:", [item["id"] for item in items])
print()
print(text)

retrieved: ['pto-2', 'pto-3', 'pto-1', 'pto-4']

After five years, you would receive 20 PTO days per year [pto-3]. To request them, you need to submit your PTO requests in advance through the HR system, unless it is related to an emergency [pto-4].
